# 623 SPP v16A: strict v15 checkpoint re-decode

v16A reuses each v15 checkpoint byte-for-byte and keeps the keyed hurdle/Poisson count decoder. Guard labels select between deterministic joint-class MAP and learned-scale-aware component-peak MAP; evaluation labels never select a mode. The lossless line58 + callback-kind encoder and chronological `DEMAND(addr)` / `CACHE_FILL(evicted_addr)` inputs are unchanged. SPP actions remain labels/comparator replay only; PC, private tables, thresholds, degree, candidate banks, page-offset classes, future rows, and raw teacher event IDs are excluded. This is matched-input open-loop replay, not a closed-loop live-NN claim.

In [ ]:
import hashlib, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
OPERATION='redecode-v16a'  # set train-v15 only to reproduce the preserved v15 workflow
RUN_ID={'train-v15':'623_offline_lstm_spp_keyed_crn_joint_fill_v15_seed7','redecode-v16a':'623_offline_lstm_spp_keyed_crn_joint_map_v16a_seed7'}[OPERATION]
DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_spp/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); assert name in uploaded,f'Select {name}'
archive=f'{DRIVE_ROOT}/{name}'; pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle: handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,item=record.split(maxsplit=1); item=item.lstrip('*')
    assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified',archive)

In [ ]:
import json
TRACE='623.xalancbmk_s-700B'; POLICY='spp'; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','teacher':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_teacher_actions.csv.gz'} for role in ROLES}
for items in INPUTS.values():
 for path in items.values(): assert os.path.isfile(path),path
manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected={'status':'PASS','experiment_revision':'spp_source_input_variable_delta_fill_feedback_free_running_v11','event_logger_schema':'623_causal_trigger_fill_v6','neural_role':'standalone_direct_action_prefetcher','source_decision_effective_external_input':['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr'],'model_input_is_causal_external_event_sequence_only':True,'cache_fill_feedback_used_as_raw_external_input':True,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':'free_running_autoregressive_same_as_inference','decoder_previous_teacher_action_used_as_input':False,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'neural_degree_cap':None,'fixed_page_offset_classes':None,'same_page_rule_used_by_neural_inference':False,'future_label_window_used':False,'fill_lead_cutoff_used':False,'inference_policy_hardcodes_used':False}
bad={k:(manifest.get(k),v) for k,v in expected.items() if manifest.get(k)!=v}; assert not bad,bad
assert manifest['training_runtime_fields']==['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr']==manifest['inference_runtime_fields']
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_spp/python/train_and_offline_infer.py'
SOURCE=f'{INPUT_DIR}/spp_source_contract.json'

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
POINTS=[(8,'p0',2682),(16,'p1',6242),(32,'p2',16050),(64,'p3',46418),(128,'p4',150162)]
prefix='guard_joint_map_spp_lstm_h' if OPERATION=='redecode-v16a' else 'joint_delta_fill_spp_lstm_h'
SPECS=[{'tag':prefix+str(size),'family':'lstm','size':size,'pair':pair,'parameters':parameters} for size,pair,parameters in POINTS]
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY,'--operation',OPERATION]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-teacher-actions',INPUTS[role]['teacher']]
 cmd += ['--source-contract',SOURCE,'--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed','7','--decoder-seed','7','--epochs','10','--chunk-len','1024','--accumulate-chunks','16']
 parent=None
 if OPERATION=='redecode-v16a':
  parent_tag=f"joint_delta_fill_spp_lstm_h{spec['size']}"; parent=f'{INPUT_DIR}/parent_checkpoints/{parent_tag}'
  cmd += ['--parent-model',f'{parent}/model.pt','--parent-metadata',f'{parent}/run_metadata.json','--parent-training-history',f'{parent}/training_history.csv','--parent-run-id','623_offline_lstm_spp_keyed_crn_joint_fill_v15_seed7']
 print('\n'+('Re-decoding' if parent else 'Training'),spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={'model_tag':spec['tag'],'model_family':'lstm','track_model_family':'lstm','model_revision':('compact_crn_joint_delta_fill_guard_map_v16a' if parent else 'compact_crn_joint_delta_fill_mixture_v15'),'parameter_count':spec['parameters'],'runtime_feature_count':59,'matched_normal_prefetcher':POLICY,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':'free_running_autoregressive_same_as_inference','decoder_previous_teacher_action_used_as_input':False,'decoder_free_running_self_test':'PASS','model_input_is_causal_external_event_sequence_only':True,'cache_fill_feedback_used_as_raw_external_input':True,'model_does_not_use_pc':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'probability_threshold_used':False,'neural_degree_cap':None,'gate_class_weighting_used':False,'gate_training_objective':'unweighted_bernoulli_nll','gate_decoding_rule':'event_keyed_bernoulli_inverse_cdf','request_count_training_objective':'unweighted_bernoulli_hurdle_plus_positive_poisson_excess_nll','request_count_decoding_rule':'event_keyed_bernoulli_plus_common_quantile_poisson_inverse_cdf','request_count_residual_scope':'none_event_local','common_random_numbers_across_capacities':True,'strict_common_random_numbers_across_capacities':True,'cross_event_rng_state_used':False,'stochastic_decoding_reproducible':True,'cross_event_probability_credit_used':False,'sampled_outputs_used_as_decoder_feedback':False,'decoder_probability_mass_carries_train_guard_history':False,'decoder_sampling_roles':(['guard','eval'] if parent else ['eval']),'joint_delta_fill_dependency_modeled':True,'joint_pair_classes':8,'joint_delta_fill_training_objective':'unweighted_joint_delta_component_fill_mixture_nll','delta_decoder_feedback_rule':'complete_joint_distribution_expectation_same_in_training_and_inference','same_source_input_offline_claim_allowed':True,'closed_loop_live_claim_allowed':False,'keyed_sampling_self_test':'PASS','joint_delta_fill_sampling_self_test':'PASS','experiment_revision':'spp_source_input_variable_delta_fill_feedback_free_running_v11'}
 if parent: expected.update({'operation':'redecode-v16a','decoder_revision':'guard_selected_deterministic_joint_map_v16a','decoder_candidate_modes':['joint_class_map','component_peak_map'],'parent_run_id':'623_offline_lstm_spp_keyed_crn_joint_fill_v15_seed7','parent_model_revision':'compact_crn_joint_delta_fill_mixture_v15','weights_model_revision':'compact_crn_joint_delta_fill_mixture_v15','weights_retrained':False,'checkpoint_reused':True,'decoder_only_change':True,'strict_checkpoint_validation_passed':True,'guard_selection_uses_eval_labels':False,'guard_selection_uses_guard_labels_only':True,'deterministic_joint_map_self_test':'PASS','fill_decoding_rule':'guard_selected_joint_pair_map','delta_mixture_decoding_rule':'guard_selected_joint_component_then_component_mean'})
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 assert meta['training_runtime_fields']==['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr']==meta['inference_runtime_fields']
 encoder_hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}; assert len(encoder_hashes)==1 and isinstance(next(iter(encoder_hashes)),str) and len(next(iter(encoder_hashes)))==64,encoder_hashes
 sampler=meta.get('decoder_sampler',{}); assert sampler.get('sampler_revision')=='sha256_event_keyed_inverse_cdf_crn_v1' and sampler.get('poisson_backend')=='scipy.stats.poisson.ppf' and sampler.get('cross_event_rng_state') is False,sampler
 if parent:
  assert meta['selected_decoder_mode'] in ('joint_class_map','component_peak_map') and meta['joint_delta_fill_decoding_rule']==meta['selected_decoder_mode']
  audit=meta['guard_decoder_selection']; assert set(audit)=={'joint_class_map','component_peak_map'} and meta['selected_decoder_mode']==max(audit,key=lambda mode:audit[mode]['selection_key'])
  assert hashlib.sha256(pathlib.Path(f'{out}/model.pt').read_bytes()).hexdigest()==hashlib.sha256(pathlib.Path(f'{parent}/model.pt').read_bytes()).hexdigest()==meta['parent_checkpoint_sha256']==meta['model_checkpoint_sha256']
  assert hashlib.sha256(pathlib.Path(f'{parent}/run_metadata.json').read_bytes()).hexdigest()==meta['parent_run_metadata_sha256']
  assert hashlib.sha256(pathlib.Path(f'{out}/training_history.csv').read_bytes()).hexdigest()==meta['parent_training_history_sha256']==meta['training_history_sha256']
 SWEEP.append({k:meta[k] for k in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','decision_rule','selected_decoder_mode','offline_normal_entries','offline_nn_entries','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
active_revision='compact_crn_joint_delta_fill_guard_map_v16a' if OPERATION=='redecode-v16a' else 'compact_crn_joint_delta_fill_mixture_v15'
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'input_revision':'spp_source_input_variable_delta_fill_feedback_free_running_v11','model_revision':active_revision,'operation':OPERATION,'decoder_seed':7,'points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')
files.download(OUTPUT_ARCHIVE)

Copy the output archive to the matching server run and launch replay. Teacher actions are supervised labels and the normal replay only; they are never neural inference inputs, gates, or budgets.